# Data Cleaning — Messy Customer Dataset

**Objective:** Take a deliberately messy dataset and systematically transform it into a clean,
analysis-ready dataset, documenting every decision along the way.

**Dataset:** Built to work with any messy tabular CSV. Search **"dirty dataset for data cleaning
practice"** on Kaggle (tags: "data cleaning") — the "Titanic dataset" or "Housing dataset" are
good real options — and place it at `data/messy_dataset.csv`.

If that file isn't found, this notebook generates a **realistic messy customer dataset** with the
exact problems this task is meant to test: inconsistent text casing, mixed date formats, missing
values, duplicate rows, wrong data types, and outliers — so every cell below runs end-to-end.

> ⚠️ For your actual submission, swap in a real Kaggle dataset — drop the CSV into
> `data/messy_dataset.csv` and re-run. Column names will differ, so tell me the real headers and
> I'll adjust the cleaning steps to match.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
np.random.seed(11)

## 1. Load Dataset

In [2]:
DATA_PATH = "data/messy_dataset.csv"

def generate_messy_dataset(n=600):
    """Generates a realistic messy customer dataset with the classic real-world problems:
    inconsistent casing, mixed date formats, missing values, duplicates, wrong dtypes, outliers."""
    genders_raw = np.random.choice(["Male","male","M","Female","female","F","FEMALE"], size=n,
                                    p=[0.22,0.1,0.08,0.22,0.1,0.08,0.2])
    countries_raw = np.random.choice(["India","india","INDIA","USA","usa","U.S.A","Germany","germany "],
                                      size=n, p=[0.25,0.1,0.05,0.2,0.1,0.05,0.15,0.1])
    ages = np.random.randint(18, 70, size=n).astype(float)

    date_formats = ["%Y-%m-%d", "%d/%m/%Y", "%B %d, %Y", "%d-%b-%Y"]
    base_dates = pd.Timestamp("2022-01-01") + pd.to_timedelta(np.random.randint(0, 900, size=n), unit="D")
    join_dates_str = [d.strftime(np.random.choice(date_formats)) for d in base_dates]

    salaries = np.round(np.random.normal(55000, 15000, size=n), 2)
    salary_str = []
    for s in salaries:
        style = np.random.choice(["plain","dollar","comma"])
        if style == "plain": salary_str.append(str(s))
        elif style == "dollar": salary_str.append(f"${s:,.2f}")
        else: salary_str.append(f"{s:,.2f}")

    df = pd.DataFrame({
        "CustomerID": np.arange(1, n+1),
        "Name": [f"Customer_{i}" for i in range(1, n+1)],
        "Gender": genders_raw,
        "Age": ages,
        "Country": countries_raw,
        "JoinDate": join_dates_str,
        "Salary": salary_str,
    })

    # inject missing values
    for col, frac in [("Age", 0.06), ("Salary", 0.05), ("Gender", 0.03), ("Country", 0.02)]:
        idx = df.sample(frac=frac, random_state=hash(col)%1000).index
        df.loc[idx, col] = np.nan

    # inject unrealistic outliers / range anomalies
    outlier_idx = df.sample(frac=0.02, random_state=42).index
    df.loc[outlier_idx, "Age"] = np.random.choice([-5, 150, 999], size=len(outlier_idx))
    salary_outlier_idx = df.sample(frac=0.015, random_state=7).index
    df.loc[salary_outlier_idx, "Salary"] = "500000.00"

    # inject duplicate rows
    dupes = df.sample(frac=0.03, random_state=3)
    df = pd.concat([df, dupes], ignore_index=True)

    return df.sample(frac=1, random_state=1).reset_index(drop=True)  # shuffle

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real dataset from {DATA_PATH}")
else:
    df = generate_messy_dataset()
    print("Real dataset not found — using generated messy dataset for demonstration.")

print("Shape:", df.shape)
df.head(10)

Real dataset not found — using generated messy dataset for demonstration.
Shape: (618, 7)


,CustomerID,Name,Gender,Age,Country,JoinDate,Salary
0,313,Customer_313,Male,20.0,Germany,19/11/2022,"$51,553.25"
1,161,Customer_161,Male,22.0,germany,"May 06, 2023","$40,094.39"
2,224,Customer_224,Female,39.0,INDIA,20-Apr-2022,"66,925.70"
3,430,Customer_430,FEMALE,64.0,USA,13/09/2023,55079.19
4,24,Customer_24,male,40.0,usa,07/01/2022,"$71,436.02"
5,48,Customer_48,FEMALE,55.0,usa,06/03/2023,53930.58
6,237,Customer_237,FEMALE,54.0,USA,18/06/2023,44129.94
7,149,Customer_149,Male,65.0,INDIA,"April 08, 2024","$52,069.90"
8,422,Customer_422,Male,NaN,USA,2023-01-29,57729.03
9,536,Customer_536,M,21.0,INDIA,"November 01, 2022",56763.54


## 2. Data Quality Report (Before Cleaning)

In [3]:
def data_quality_report(data, label=""):
    report = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "null_count": data.isnull().sum(),
        "null_pct": (data.isnull().sum() / len(data) * 100).round(1),
        "unique_values": data.nunique(),
    })
    print(f"--- Data Quality Report {label} ---")
    print(f"Row count: {len(data)}")
    print(f"Duplicate rows: {data.duplicated().sum()}")
    return report

report_before = data_quality_report(df, "(BEFORE cleaning)")
report_before

--- Data Quality Report (BEFORE cleaning) ---
Row count: 618
Duplicate rows: 18


,dtype,null_count,null_pct,unique_values
CustomerID,int64,0,0.0,600
Name,object,0,0.0,600
Gender,object,19,3.1,7
Age,float64,36,5.8,55
Country,object,13,2.1,8
JoinDate,object,0,0.0,557
Salary,object,31,5.0,563


In [4]:
# Inspect the actual inconsistent values before deciding how to fix them
print("Unique Gender values:", df["Gender"].unique())
print("\nUnique Country values:", df["Country"].unique())
print("\nSample JoinDate values:", df["JoinDate"].dropna().sample(5, random_state=1).tolist())
print("\nSample Salary values:", df["Salary"].dropna().sample(5, random_state=1).tolist())
print("\nAge range anomalies:", sorted(df["Age"].dropna().unique())[:5], "...", sorted(df["Age"].dropna().unique())[-5:])

Unique Gender values: ['Male' 'Female' 'FEMALE' 'male' 'M' 'F' 'female' nan]

Unique Country values: ['Germany' 'germany ' 'INDIA' 'USA' 'usa' 'India' nan 'india' 'U.S.A']

Sample JoinDate values: ['June 25, 2022', '08/02/2023', '2023-06-20', 'December 29, 2023', '23-Oct-2022']

Sample Salary values: ['17710.6', '47076.87', '84144.34', '52654.12', '$57,120.22']

Age range anomalies: [np.float64(-5.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0)] ... [np.float64(67.0), np.float64(68.0), np.float64(69.0), np.float64(150.0), np.float64(999.0)]


**Observations from the raw data:**
- `Gender` has 7 inconsistent variants of just two categories (Male/Female)
- `Country` has inconsistent casing and stray whitespace
- `JoinDate` is stored as text in at least 4 different formats
- `Salary` is stored as text, sometimes with `$` and thousands-separator commas
- `Age` contains impossible values (negative ages, ages like 150/999) — clear data entry errors, not real outliers
- There are missing values in `Age`, `Salary`, `Gender`, and `Country`
- There are duplicate rows

## 3. Duplicate Removal

In [5]:
dupe_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {dupe_count} duplicate row(s). New shape: {df.shape}")

Removed 18 duplicate row(s). New shape: (600, 7)


## 4. Standardisation — Fixing Inconsistent Formatting

In [6]:
# Gender: collapse all variants into 'Male' / 'Female'
gender_map = {
    "male": "Male", "m": "Male", "Male": "Male",
    "female": "Female", "f": "Female", "Female": "Female", "FEMALE": "Female",
}
df["Gender"] = df["Gender"].str.strip().map(lambda x: gender_map.get(x, gender_map.get(str(x).lower(), x)) if pd.notna(x) else x)
print("Gender values after standardisation:", df["Gender"].unique())

Gender values after standardisation: ['Male' 'Female' nan]


In [7]:
# Country: strip whitespace, standardise casing to Title Case, fix known acronyms
df["Country"] = df["Country"].str.strip().str.title()
df["Country"] = df["Country"].replace({"Usa": "USA", "U.S.A": "USA", "India": "India"})
print("Country values after standardisation:", df["Country"].unique())

Country values after standardisation: ['Germany' 'India' 'USA' nan]


In [8]:
# JoinDate: parse the mixed formats into a single consistent datetime column
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%B %d, %Y", "%d-%b-%Y"]:
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(value, errors="coerce")  # last-resort fallback

df["JoinDate"] = df["JoinDate"].apply(parse_mixed_date)
print("JoinDate dtype now:", df["JoinDate"].dtype)
print("Unparseable dates remaining:", df["JoinDate"].isna().sum())
df[["JoinDate"]].head()

JoinDate dtype now: datetime64[ns]
Unparseable dates remaining: 0


,JoinDate
0,2022-11-19
1,2023-05-06
2,2022-04-20
3,2023-09-13
4,2022-01-07


In [9]:
# Salary: strip $ and commas, convert to float
df["Salary"] = (
    df["Salary"].astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .replace("nan", np.nan)
    .astype(float)
)
print("Salary dtype now:", df["Salary"].dtype)
df["Salary"].describe()

Salary dtype now: float64


count       571.000000
mean      60459.447828
std       57551.676191
min        9675.150000
25%       43642.035000
50%       52707.350000
75%       63294.615000
max      500000.000000
Name: Salary, dtype: float64

## 5. Missing Data Handling

Each column gets a strategy justified by what the column represents:

- **Age** → median imputation. Age is roughly symmetric with a few extreme entry errors (handled
  separately as outliers below), so the median is robust and preserves the distribution better than
  the mean.
- **Salary** → median imputation. Salary distributions are typically right-skewed; median avoids
  being pulled by high earners.
- **Gender** → fill with `"Unknown"` rather than guessing. Imputing a demographic category from
  nothing is not justifiable — better to be explicit that it's missing.
- **Country** → fill with `"Unknown"` for the same reason as Gender.

In [10]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Salary"] = df["Salary"].fillna(df["Salary"].median())
df["Gender"] = df["Gender"].fillna("Unknown")
df["Country"] = df["Country"].fillna("Unknown")

print("Remaining nulls per column:\n", df.isnull().sum())

Remaining nulls per column:
 CustomerID    0
Name          0
Gender        0
Age           0
Country       0
JoinDate      0
Salary        0
dtype: int64


## 6. Outlier Detection — IQR Method

In [11]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

age_low, age_high = iqr_bounds(df["Age"])
print(f"Age IQR bounds: [{age_low:.1f}, {age_high:.1f}]")
age_outliers = df[(df["Age"] < age_low) | (df["Age"] > age_high)]
print(f"Age outliers found: {len(age_outliers)}")
print(age_outliers["Age"].tolist()[:10])

Age IQR bounds: [-4.0, 92.0]
Age outliers found: 12
[-5.0, -5.0, -5.0, 999.0, 999.0, 150.0, 999.0, 150.0, 150.0, 150.0]


**Decision on Age outliers: remove, don't cap.** Values like `-5`, `150`, or `999` are not
legitimate extreme ages — they're data entry errors (a human or system clearly typed garbage).
Capping them to the IQR boundary would silently invent a plausible-looking but fabricated age.
Since these rows have no way to recover the true age, the affected rows are dropped entirely.

In [12]:
before_rows = len(df)
df = df[(df["Age"] >= age_low) & (df["Age"] <= age_high)].reset_index(drop=True)
print(f"Dropped {before_rows - len(df)} row(s) with invalid Age. New shape: {df.shape}")

Dropped 12 row(s) with invalid Age. New shape: (588, 7)


In [13]:
sal_low, sal_high = iqr_bounds(df["Salary"])
print(f"Salary IQR bounds: [{sal_low:.1f}, {sal_high:.1f}]")
salary_outliers = df[(df["Salary"] < sal_low) | (df["Salary"] > sal_high)]
print(f"Salary outliers found: {len(salary_outliers)}")

Salary IQR bounds: [17308.5, 89027.3]
Salary outliers found: 18


**Decision on Salary outliers: cap, don't remove.** Unlike the impossible ages, a very high salary
(e.g. `$500,000`) is *plausible* — it could be a genuine senior employee. Removing these rows would
throw away real customers. Capping (winsorizing) to the IQR boundary keeps the row but limits its
influence on downstream statistics like the mean.

In [14]:
df["Salary"] = df["Salary"].clip(lower=sal_low, upper=sal_high)
print("Salary stats after capping:\n", df["Salary"].describe())

Salary stats after capping:
 count      588.000000
mean     53864.038890
std      14662.405863
min      17308.453750
25%      44203.030000
50%      52707.350000
75%      62132.747500
max      89027.323750
Name: Salary, dtype: float64


## 7. Data Type Correction

In [15]:
df["CustomerID"] = df["CustomerID"].astype(str)
df["Age"] = df["Age"].astype(int)
df["Salary"] = df["Salary"].astype(float).round(2)
# JoinDate is already datetime from Section 4

print(df.dtypes)

CustomerID            object
Name                  object
Gender                object
Age                    int64
Country               object
JoinDate      datetime64[ns]
Salary               float64
dtype: object


## 8. Before vs. After Summary

In [16]:
report_after = data_quality_report(df, "(AFTER cleaning)")

summary = pd.DataFrame({
    "Metric": ["Row count", "Duplicate rows", "Total null values", "Correct dtypes"],
    "Before": [len(report_before), dupe_count, int(report_before["null_count"].sum()), "No (mixed strings)"],
    "After": [len(df), df.duplicated().sum(), int(df.isnull().sum().sum()), "Yes"],
})
summary

--- Data Quality Report (AFTER cleaning) ---
Row count: 588
Duplicate rows: 0


,Metric,Before,After
0,Row count,7,588
1,Duplicate rows,18,0
2,Total null values,99,0
3,Correct dtypes,No (mixed strings),Yes


## 9. Save the Cleaned Dataset

In [17]:
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/cleaned_dataset.csv", index=False)
print("Saved outputs/cleaned_dataset.csv")
print("Final shape:", df.shape)
df.head(10)

Saved outputs/cleaned_dataset.csv
Final shape: (588, 7)


,CustomerID,Name,Gender,Age,Country,JoinDate,Salary
0,313,Customer_313,Male,20,Germany,2022-11-19,51553.25
1,161,Customer_161,Male,22,Germany,2023-05-06,40094.39
2,224,Customer_224,Female,39,India,2022-04-20,66925.70
3,430,Customer_430,Female,64,USA,2023-09-13,55079.19
4,24,Customer_24,Male,40,USA,2022-01-07,71436.02
5,48,Customer_48,Female,55,USA,2023-03-06,53930.58
6,237,Customer_237,Female,54,USA,2023-06-18,44129.94
7,149,Customer_149,Male,65,India,2024-04-08,52069.90
8,422,Customer_422,Male,44,USA,2023-01-29,57729.03
9,536,Customer_536,Male,21,India,2022-11-01,56763.54


## Conclusion

Starting from a messy 600+ row dataset with inconsistent text formatting, mixed date formats,
missing values, duplicates, wrong data types, and both fabricated and legitimate outliers, this
notebook produced a clean, analysis-ready dataset:

- 0 duplicate rows
- 0 missing values
- Consistent categorical formatting (Gender, Country)
- A single datetime format for JoinDate
- Numeric columns properly typed
- Outliers handled with a documented, column-specific rationale rather than a blanket rule

Every cleaning decision above is justified based on what each column represents, not applied
mechanically — that judgment is the actual skill this task is testing.